In [ ]:
# ============================================
# Non-Deterministic Deep Clustering with AutoVaDE-DP in TensorFlow
# ============================================
# - VAE deep clustering with truncated DP-GMM prior
# - β-annealing, early stopping
# - Baselines: PCA+k-means, AE+k-means
# - Uncertainty: MC-Dropout
# - Calibration: temperature scaling via self-consistency
# - Visualizations: UMAP and t-SNE (latent space), sample reconstructions
# ---------------------------------------------------------------

!pip -q install tensorflow==2.12 scikit-learn umap-learn tqdm pandas matplotlib --upgrade

# 1) Imports & Config
import os, json, math, time, random
import numpy as np
import tensorflow as tf
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
from sklearn.manifold import TSNE
import umap

# Repro
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)

class CFG:
    dataset = "mnist"
    seed = 42
    save_dir = "/content/autovade_dp_runs"
    # model
    latent_dim = 16
    k_max = 20
    alpha = 1.0                  # DP Beta(1, alpha)
    dropout = 0.3                # MC-dropout in encoder/decoder
    # training
    epochs = 50
    batch_size = 256
    lr = 3e-4
    beta = 1.0
    beta_anneal_epochs = 20
    early_stop = True
    patience = 15
    # eval
    mc_T = 20                    # MC-Dropout passes
    make_plots = True            # UMAP/t-SNE & recon samples
    # baseline
    run_baselines = True

os.makedirs(CFG.save_dir, exist_ok=True)
set_seed(CFG.seed)

# 2) Data (MNIST / Fashion-MNIST)
from tensorflow.keras.datasets import mnist, fashion_mnist
def load_dataset(name="mnist"):
    if name.lower() == "mnist":
        (x_train, y_train), (x_test, y_test) = mnist.load_data()
    else:
        (x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()
    x_train = x_train.astype("float32")/255.0
    x_test = x_test.astype("float32")/255.0
    x_train = np.expand_dims(x_train, -1)
    x_test = np.expand_dims(x_test, -1)
    return (x_train, y_train), (x_test, y_test)

(x_train, y_train), (x_test, y_test) = load_dataset(CFG.dataset)
img_shape = x_train.shape[1:]
train_ds = tf.data.Dataset.from_tensor_slices(x_train).shuffle(10000, seed=CFG.seed).batch(CFG.batch_size).prefetch(tf.data.AUTOTUNE)

# 3) DP stick-breaking & logpdf utilities
@tf.function
def stick_breaking_weights(v):  # v in (0,1) [K]
    one_minus_v = 1.0 - v
    prefix = tf.math.cumprod(tf.concat([tf.ones_like(v[:1]), one_minus_v[:-1]], axis=0))
    pi = v * prefix
    return pi / (tf.reduce_sum(pi) + 1e-8)

def dp_v_prior_loss(v_logits, alpha=1.0):
    v = tf.sigmoid(v_logits)
    v = tf.clip_by_value(v, 1e-6, 1.0-1e-6)
    logp = tf.math.log(alpha) + (alpha-1.0)*tf.math.log(1.0-v)  # Beta(1,alpha)
    return -tf.reduce_sum(logp)

def log_mix_gaussians(z, means, logvars, pi):
    # z:[B,D], means/logvars:[K,D], pi:[K]
    D = tf.cast(tf.shape(z)[-1], z.dtype)
    log_probs = tf.math.log(pi+1e-12) - 0.5*(D*tf.math.log(2*np.pi)
                 + tf.reduce_sum(logvars, axis=-1)
                 + tf.reduce_sum((tf.expand_dims(z,-2)-means)**2/tf.exp(logvars), axis=-1))
    m = tf.reduce_max(log_probs, axis=-1, keepdims=True)
    return tf.squeeze(m + tf.math.log(tf.reduce_sum(tf.exp(log_probs-m), axis=-1, keepdims=True)), -1)

# 4) Model (Encoder / Decoder / AutoVaDE-DP)
from tensorflow.keras import layers as L

class Encoder(tf.keras.Model):
    def __init__(self, latent_dim=16, dropout=0.3):
        super().__init__()
        self.c1 = L.Conv2D(32, 3, padding='same', use_bias=False)
        self.b1 = L.BatchNormalization()
        self.c2 = L.Conv2D(64, 3, strides=2, padding='same', use_bias=False)
        self.b2 = L.BatchNormalization()
        self.c3 = L.Conv2D(128, 3, strides=2, padding='same', use_bias=False)
        self.b3 = L.BatchNormalization()
        self.flt = L.Flatten()
        self.fc = L.Dense(256, activation='relu')
        self.mu = L.Dense(latent_dim)
        self.logvar = L.Dense(latent_dim)
        self.drop = L.Dropout(dropout)

    def call(self, x, training=True):
        x = self.drop(tf.nn.relu(self.b1(self.c1(x), training=training)), training=training)
        x = self.drop(tf.nn.relu(self.b2(self.c2(x), training=training)), training=training)
        x = self.drop(tf.nn.relu(self.b3(self.c3(x), training=training)), training=training)
        x = self.fc(self.flt(x))
        return self.mu(x), self.logvar(x)

class Decoder(tf.keras.Model):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.fc = L.Dense(7*7*128, activation='relu')
        self.rs = L.Reshape((7,7,128))
        self.d1 = L.Conv2DTranspose(64, 3, strides=2, padding='same', use_bias=False)
        self.b1 = L.BatchNormalization()
        self.d2 = L.Conv2DTranspose(32, 3, strides=2, padding='same', use_bias=False)
        self.b2 = L.BatchNormalization()
        self.out_logits = L.Conv2D(1, 3, padding='same')
        self.drop = L.Dropout(0.3)

    def call(self, z, training=True):
        x = self.rs(self.fc(z))
        x = self.drop(tf.nn.relu(self.b1(self.d1(x), training=training)), training=training)
        x = self.drop(tf.nn.relu(self.b2(self.d2(x), training=training)), training=training)
        return self.out_logits(x)

class AutoVaDE_DP(tf.keras.Model):
    def __init__(self, latent_dim=16, k_max=20, alpha=1.0, dropout=0.3):
        super().__init__()
        self.encoder = Encoder(latent_dim, dropout=dropout)
        self.decoder = Decoder(dropout=dropout)
        self.v_logits = tf.Variable(tf.zeros([k_max]), name="v_logits")
        self.comp_mu = tf.Variable(tf.random.normal([k_max, latent_dim], stddev=0.1), name="comp_mu")
        self.comp_logvar = tf.Variable(tf.zeros([k_max, latent_dim]), name="comp_logvar")
        self.alpha = alpha

    def sample_q(self, mu, logvar):
        eps = tf.random.normal(tf.shape(mu))
        return mu + tf.exp(0.5*logvar)*eps

    def bernoulli_recon(self, x, logits):
        bce = tf.nn.sigmoid_cross_entropy_with_logits(labels=x, logits=logits)
        return tf.reduce_mean(tf.reduce_sum(bce, axis=[1,2,3]))

    def kl_qz_pz(self, mu, logvar, z):
        # log q(z|x)
        D = tf.cast(tf.shape(mu)[1], mu.dtype)
        log_q = -0.5*(D*tf.math.log(2*np.pi)
                      + tf.reduce_sum(logvar, axis=-1)
                      + tf.reduce_sum(((z-mu)**2)/tf.exp(logvar), axis=-1))
        # log p(z) mixture
        v = tf.sigmoid(self.v_logits)
        pi = stick_breaking_weights(v)
        log_p = log_mix_gaussians(z, self.comp_mu, self.comp_logvar, pi)
        return tf.reduce_mean(log_q - log_p)

    def responsibilities(self, z):
        v = tf.sigmoid(self.v_logits)
        pi = stick_breaking_weights(v)
        diff = tf.expand_dims(z,1) - tf.expand_dims(self.comp_mu,0)  # [B,K,D]
        logp = -0.5*(tf.cast(tf.shape(z)[-1], z.dtype)*tf.math.log(2*np.pi)
                     + tf.reduce_sum(self.comp_logvar, axis=-1)[None,:]
                     + tf.reduce_sum(diff**2/tf.exp(self.comp_logvar)[None,:,:], axis=-1))  # [B,K]
        logw = tf.math.log(pi+1e-12)[None,:] + logp
        m = tf.reduce_max(logw, axis=-1, keepdims=True)
        r = tf.exp(logw - m)
        return r / tf.reduce_sum(r, axis=-1, keepdims=True)

    def step(self, x, beta=1.0, dp_reg_scale=1.0, comp_reg_scale=1e-3):
        mu, logvar = self.encoder(x, training=True)
        z = self.sample_q(mu, logvar)
        x_logits = self.decoder(z, training=True)
        recon = self.bernoulli_recon(x, x_logits)
        kl = self.kl_qz_pz(mu, logvar, z)
        dp_reg = dp_v_prior_loss(self.v_logits, alpha=self.alpha)
        comp_reg = tf.reduce_sum(self.comp_mu**2) + tf.reduce_sum(self.comp_logvar**2)
        total = recon + beta*kl + dp_reg_scale*dp_reg + comp_reg_scale*comp_reg
        return total, {"recon":recon, "kl":kl, "dp_reg":dp_reg, "comp_reg":comp_reg}

# 5) Training loop
model = AutoVaDE_DP(latent_dim=CFG.latent_dim, k_max=CFG.k_max, alpha=CFG.alpha, dropout=CFG.dropout)
opt = tf.keras.optimizers.Adam(CFG.lr)

best_loss = float("inf"); wait=0
history = []

print("Training...")
for epoch in range(CFG.epochs):
    betat = min(1.0, (epoch+1)/max(1, CFG.beta_anneal_epochs)) * CFG.beta
    batch_losses = []
    for xb in train_ds:
        with tf.GradientTape() as tape:
            total, parts = model.step(xb, beta=betat)
        grads = tape.gradient(total, model.trainable_variables)
        opt.apply_gradients(zip(grads, model.trainable_variables))
        batch_losses.append([total.numpy(), parts["recon"].numpy(), parts["kl"].numpy()])
    ep = np.mean(batch_losses, axis=0)
    history.append(ep.tolist())
    if (epoch+1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:03d} | total={ep[0]:.3f} recon={ep[1]:.3f} kl={ep[2]:.3f}")

    if CFG.early_stop:
        if ep[0] < best_loss - 1e-4:
            best_loss = ep[0]; wait=0
        else:
            wait += 1
            if wait >= CFG.patience:
                print("Early stopping triggered.")
                break

# 6) Inference helpers
def infer_mu(model, X, batch=512, training=False):
    out=[]
    for i in range(0, len(X), batch):
        xb = X[i:i+batch]
        mu, logvar = model.encoder(xb, training=training)
        out.append(mu.numpy())
    return np.concatenate(out, axis=0)

def soft_assign(model, X, batch=512, training=False, sample=True):
    R=[]
    for i in range(0, len(X), batch):
        xb = X[i:i+batch]
        mu, logvar = model.encoder(xb, training=training)
        if sample:
            eps = np.random.randn(*mu.shape).astype(np.float32)
            z = mu + np.exp(0.5*logvar) * eps
        else:
            z = mu
        r = model.responsibilities(z).numpy()
        R.append(r)
    return np.concatenate(R, axis=0)

# 7) Clustering metrics + active-K
Z = infer_mu(model, x_test, training=False)
R = soft_assign(model, x_test, training=False, sample=True)
y_pred = np.argmax(R, axis=-1)

def cluster_metrics(y_true, y_pred, Z=None):
    ari = adjusted_rand_score(y_true, y_pred)
    nmi = normalized_mutual_info_score(y_true, y_pred)
    sil = None
    if Z is not None and len(np.unique(y_pred)) > 1:
        try:
            sil = silhouette_score(Z, y_pred)
        except Exception:
            sil = None
    return {"ARI": float(ari), "NMI": float(nmi), "Silhouette": (None if sil is None else float(sil))}

metrics = cluster_metrics(y_test, y_pred, Z)
print("Clustering metrics:", metrics)

v = tf.sigmoid(model.v_logits).numpy()
# normalize to sum 1 (safe)
pi = (v * np.concatenate([[1.0], np.cumprod(1.0 - v[:-1])]))
pi = pi / pi.sum()
active = (pi > 0.01).sum()
print(f"Active clusters (pi>0.01): {active}/{len(pi)}")
print("pi:", np.round(pi, 4))

# 8) MC-Dropout uncertainty
def mc_dropout_probs(model, X, T=20, batch=512):
    probs_T=[]
    for t in range(T):
        R = soft_assign(model, X, batch=batch, training=True, sample=True)  # training=True keeps dropout active
        probs_T.append(R)
    return np.stack(probs_T, axis=0)

def predictive_entropy(p, eps=1e-12):
    p = np.clip(p, eps, 1.0)
    return -np.sum(p*np.log(p), axis=-1)

def mutual_information(probs_T):
    p_mean = probs_T.mean(axis=0)
    H_mean = predictive_entropy(p_mean)
    H_exp = np.mean([predictive_entropy(p) for p in probs_T], axis=0)
    return H_mean - H_exp

print("Estimating uncertainty with MC-Dropout...")
probs_T = mc_dropout_probs(model, x_test, T=CFG.mc_T)
ent = predictive_entropy(probs_T.mean(axis=0)).mean()
mi = mutual_information(probs_T).mean()
print(f"Mean predictive entropy: {ent:.3f} | Mean MI (epistemic): {mi:.3f}")

# 9) Temperature calibration (self-consistency)
def fit_temperature(logits1, logits2, steps=200, lr=1e-2):
    logT = tf.Variable(tf.math.log(tf.constant(1.0, dtype=tf.float32)))
    opt = tf.keras.optimizers.Adam(lr)
    for _ in range(steps):
        with tf.GradientTape() as tape:
            T = tf.exp(logT)
            s1 = logits1 / T
            s2 = logits2 / T
            p1 = tf.nn.softmax(s1, axis=-1)
            logp2 = tf.nn.log_softmax(s2, axis=-1)
            loss = -tf.reduce_mean(tf.reduce_sum(p1 * logp2, axis=-1))
        grads = tape.gradient(loss, [logT])
        opt.apply_gradients(zip(grads, [logT]))
    return float(tf.exp(logT).numpy())

# Use 2 MC passes as two "views"
logits1 = np.log(np.clip(probs_T[0], 1e-8, 1.0)).astype(np.float32)
logits2 = np.log(np.clip(probs_T[1], 1e-8, 1.0)).astype(np.float32)
T_star = fit_temperature(logits1, logits2, steps=200, lr=1e-2)
print(f"Optimal temperature T*: {T_star:.3f}")

# 10) Baselines (PCA+k-means, AE+k-means)
baseline_results = {}
if CFG.run_baselines:
    print("Running PCA+k-means baseline...")
    Xflat = x_train.reshape(len(x_train), -1)
    pca = PCA(n_components=50, random_state=CFG.seed).fit(Xflat)
    Zp = pca.transform(x_test.reshape(len(x_test), -1))
    km = KMeans(n_clusters=10, n_init=10, random_state=CFG.seed).fit(Zp)
    yb = km.labels_
    baseline_results["PCA+kmeans"] = {
        "ARI": float(adjusted_rand_score(y_test, yb)),
        "NMI": float(normalized_mutual_info_score(y_test, yb))
    }
    print("PCA+kmeans:", baseline_results["PCA+kmeans"])

    print("Running AE+k-means baseline...")
    # Simple AE
    inp = tf.keras.Input(shape=img_shape)
    x = L.Conv2D(32,3,padding='same',activation='relu')(inp)
    x = L.MaxPool2D()(x)
    x = L.Conv2D(64,3,padding='same',activation='relu')(x)
    x = L.MaxPool2D()(x)
    x = L.Flatten()(x)
    z = L.Dense(64,activation='relu')(x)
    e = tf.keras.Model(inp, z)
    d_in = tf.keras.Input(shape=(64,))
    y = L.Dense(7*7*64, activation='relu')(d_in)
    y = L.Reshape((7,7,64))(y)
    y = L.Conv2DTranspose(64,3,strides=2,padding='same',activation='relu')(y)
    y = L.Conv2DTranspose(32,3,strides=2,padding='same',activation='relu')(y)
    out = L.Conv2D(1,3,padding='same',activation='sigmoid')(y)
    d = tf.keras.Model(d_in, out)
    ae_in = tf.keras.Input(shape=img_shape)
    zz = e(ae_in)
    out_ae = d(zz)
    ae = tf.keras.Model(ae_in, out_ae)
    ae.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy')
    ae.fit(x_train, x_train, epochs=5, batch_size=256, verbose=0)
    Zae = e.predict(x_test, batch_size=512, verbose=0)
    km2 = KMeans(n_clusters=10, n_init=10, random_state=CFG.seed).fit(Zae)
    yb2 = km2.labels_
    baseline_results["AE+kmeans"] = {
        "ARI": float(adjusted_rand_score(y_test, yb2)),
        "NMI": float(normalized_mutual_info_score(y_test, yb2))
    }
    print("AE+kmeans:", baseline_results["AE+kmeans"])

# 11) Visualizations (optional)
def plot_umap_tsne(Z, labels, title_prefix, save_dir):
    try:
        reducer = umap.UMAP(random_state=CFG.seed)
        Zu = reducer.fit_transform(Z)
        plt.figure(figsize=(6,5))
        plt.scatter(Zu[:,0], Zu[:,1], c=labels, s=3, cmap='tab10')
        plt.title(f"{title_prefix} UMAP")
        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, f"{title_prefix.lower().replace(' ','_')}_umap.png"), dpi=150)
        plt.show()
    except Exception as e:
        print("UMAP failed:", e)

    try:
        Zt = TSNE(n_components=2, init='pca', random_state=CFG.seed).fit_transform(Z)
        plt.figure(figsize=(6,5))
        plt.scatter(Zt[:,0], Zt[:,1], c=labels, s=3, cmap='tab10')
        plt.title(f"{title_prefix} t-SNE")
        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, f"{title_prefix.lower().replace(' ','_')}_tsne.png"), dpi=150)
        plt.show()
    except Exception as e:
        print("t-SNE failed:", e)

def show_reconstructions(model, X, n=8, save_path=None):
    idx = np.random.choice(len(X), size=n, replace=False)
    x = X[idx]
    mu, logvar = model.encoder(x, training=False)
    z = mu.numpy()
    xhat = tf.sigmoid(model.decoder(z, training=False)).numpy()
    fig, axes = plt.subplots(2, n, figsize=(1.8*n, 3.2))
    for i in range(n):
        axes[0,i].imshow(x[i].squeeze(), cmap='gray'); axes[0,i].axis('off')
        axes[1,i].imshow(xhat[i].squeeze(), cmap='gray'); axes[1,i].axis('off')
    plt.suptitle("Originals (top) vs Reconstructions (bottom)")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()

if CFG.make_plots:
    plot_umap_tsne(Z, y_test, "Latent μ colored by true labels", CFG.save_dir)
    plot_umap_tsne(Z, y_pred, "Latent μ colored by predicted clusters", CFG.save_dir)
    show_reconstructions(model, x_test, n=10, save_path=os.path.join(CFG.save_dir, "recon.png"))

# 12) Save artifacts (metrics, arrays)
all_metrics = {
    "model": {
        "dataset": CFG.dataset,
        "latent_dim": CFG.latent_dim,
        "k_max": CFG.k_max,
        "alpha": CFG.alpha,
        "dropout": CFG.dropout,
        "epochs": CFG.epochs,
        "beta": CFG.beta,
        "beta_anneal_epochs": CFG.beta_anneal_epochs
    },
    "clustering": metrics,
    "dp": {
        "active_over_1pct": int(active),
        "pi": [float(v) for v in pi.tolist()]
    },
    "uncertainty": {
        "mean_predictive_entropy": float(ent),
        "mean_mutual_information": float(mi),
        "mc_T": CFG.mc_T,
        "temperature_star": float(T_star)
    },
    "baselines": baseline_results
}

np.save(os.path.join(CFG.save_dir, "Z.npy"), Z)
np.save(os.path.join(CFG.save_dir, "R.npy"), R)
np.save(os.path.join(CFG.save_dir, "pi.npy"), pi)
with open(os.path.join(CFG.save_dir, "metrics.json"), "w") as f:
    json.dump(all_metrics, f, indent=2)

print("\nSaved artifacts to:", CFG.save_dir)
print("Done ✅")


ERROR: Could not find a version that satisfies the requirement tensorflow==2.12 (from versions: 2.16.0rc0, 2.16.1, 2.16.2, 2.17.0rc0, 2.17.0rc1, 2.17.0, 2.17.1, 2.18.0rc0, 2.18.0rc1, 2.18.0rc2, 2.18.0, 2.18.1, 2.19.0rc0, 2.19.0, 2.19.1, 2.20.0rc0, 2.20.0, 2.21.0rc0)
ERROR: No matching distribution found for tensorflow==2.12


In [ ]:
# Zip everything inside the results folder
import shutil, os
from google.colab import files

shutil.make_archive("autovade_dp_runs", 'zip', CFG.save_dir)

# Download the zip
files.download("autovade_dp_runs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>